# Fine-tune Qwen2.5-0.5B on Ultron AI Chat dataset

Trains a LoRA adapter on `data/dataset_generated.jsonl` from the Ultron_AI_Chat repo.

**Setup:** Runtime -> Change runtime type -> T4 GPU

Run cells top to bottom. Do not re-run the pip install cell more than once per session.


In [ ]:
!nvidia-smi


In [ ]:
!pip install -q -U transformers datasets trl peft accelerate bitsandbytes
# Ignore any cuDF / pyarrow warning - it does not affect training.


In [ ]:
from huggingface_hub import notebook_login
notebook_login()


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
!git clone https://github.com/huntonigiri339-rgb/Ultron_AI_Chat.git

import json

DATA_PATH = "Ultron_AI_Chat/data/dataset_generated.jsonl"

with open(DATA_PATH, "r", encoding="utf-8") as f:
    raw = [json.loads(line) for line in f if line.strip()]

print(f"Loaded {len(raw)} examples")
print(raw[0])


In [ ]:
from transformers import AutoTokenizer

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

def format_example(example):
    return tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )

texts = [format_example(ex) for ex in raw]
print(texts[0][:500])


In [ ]:
from datasets import Dataset

ds = Dataset.from_dict({"text": texts})
ds = ds.train_test_split(test_size=0.1, seed=42)
print(ds)


In [ ]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)


In [ ]:
import inspect
from trl import SFTConfig

OUTPUT_DIR = "/content/drive/MyDrive/qwen-0.5b-ultron-lora"
supports = set(inspect.signature(SFTConfig.__init__).parameters.keys())

kwargs = dict(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    logging_steps=10,
    save_strategy="steps",
    save_steps=20,
    save_total_limit=2,
    fp16=True,
    bf16=False,
    report_to="none",
)

if "eval_strategy" in supports:
    kwargs["eval_strategy"] = "steps"
elif "evaluation_strategy" in supports:
    kwargs["evaluation_strategy"] = "steps"

if "max_length" in supports:
    kwargs["max_length"] = 1024
elif "max_seq_length" in supports:
    kwargs["max_seq_length"] = 1024

if "dataset_text_field" in supports:
    kwargs["dataset_text_field"] = "text"

if "packing" in supports:
    kwargs["packing"] = False

sft_config = SFTConfig(**kwargs)
print("SFTConfig built OK")


In [ ]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=ds["train"],
    eval_dataset=ds["test"],
    peft_config=lora_config,
)

cast_count = 0
for name, param in trainer.model.named_parameters():
    if "lora_" in name and param.dtype == torch.bfloat16:
        param.data = param.data.to(torch.float32)
        cast_count += 1
print(f"Casted {cast_count} LoRA tensors to fp32")


In [ ]:
trainer.train()


In [ ]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("Saved to", OUTPUT_DIR)


In [ ]:
def ask(question, system="You are a patient tutor who explains AI concepts clearly and simply."):
    msgs = [
        {"role": "system", "content": system},
        {"role": "user", "content": question},
    ]
    prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=200, do_sample=False)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

print(ask("What is a token?"))
print("---")
print(ask("Why do language models hallucinate?"))


## Reload the adapter in a future session

If Colab disconnects and you come back later:
1. Run the pip install cell again (fresh session has no packages).
2. Run this cell to reload the saved adapter.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
ADAPTER = "/content/drive/MyDrive/qwen-0.5b-ultron-lora"

tokenizer = AutoTokenizer.from_pretrained(ADAPTER)
base = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype="float16", device_map="auto")
model = PeftModel.from_pretrained(base, ADAPTER)
model.eval()
print("Adapter reloaded")
